In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig
from peft import PeftModel

BASE = "ku-nlp/deberta-v3-base-japanese"
ADAPTER_REPO = "kkatodus/text_labeller_twitter-deberta-v3-base-japanese"

# Load config from the adapter repo (so num_labels/id2label/label2id come from what you saved)
cfg = AutoConfig.from_pretrained(ADAPTER_REPO)

tokenizer = AutoTokenizer.from_pretrained(BASE, use_fast=False, trust_remote_code=False)

base = AutoModelForSequenceClassification.from_pretrained(
    BASE,
    num_labels=cfg.num_labels,          # important
    device_map="auto",
)
base.config.problem_type = "multi_label_classification"
base.config.id2label = cfg.id2label
base.config.label2id = cfg.label2id

model = PeftModel.from_pretrained(base, ADAPTER_REPO)
model.eval()

@torch.no_grad()
def predict(text: str, thr: float = 0.5):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    logits = model(**inputs).logits[0]
    probs = torch.sigmoid(logits).float().cpu().numpy()

    labels = [base.config.id2label[i] for i in range(cfg.num_labels)]
    picked = [labels[i] for i, p in enumerate(probs) if p >= thr]
    return probs, picked

probs, picked = predict("民主主義は大事ですの話をします。")
print(picked)

In [46]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig
from peft import PeftModel

BASE = "ku-nlp/deberta-v3-base-japanese"
ADAPTER_REPO = "kkatodus/text_labeller_twitter-deberta-v3-base-japanese"

# Load config from the adapter repo (so num_labels/id2label/label2id come from what you saved)
cfg = AutoConfig.from_pretrained(ADAPTER_REPO)

tokenizer = AutoTokenizer.from_pretrained(BASE, use_fast=False, trust_remote_code=False)

base = AutoModelForSequenceClassification.from_pretrained(
    BASE,
    num_labels=cfg.num_labels,          # important
    device_map="auto",
)
base.config.problem_type = "multi_label_classification"
base.config.id2label = cfg.id2label
base.config.label2id = cfg.label2id

model = PeftModel.from_pretrained(base, ADAPTER_REPO)
model.eval()

@torch.no_grad()
def predict(text: str, thr: float = 0.5):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    logits = model(**inputs).logits[0]
    probs = torch.sigmoid(logits).float().cpu().numpy()

    labels = [base.config.id2label[i] for i in range(cfg.num_labels)]
    picked = labels[probs.argmax()]
    return probs, picked

probs, picked = predict("RT@nhk_sports:／#NHK杯フィギュア#男子シングル中継解説の#本田武史さん注目は？＼放送予定は👇の特設サイトから！https://t.co/7qd0FdLIHp本田さんのインタビュー記事は👇のリンクから！https://t.co/y2…")
print(picked)

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at ku-nlp/deberta-v3-base-japanese and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


その他


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig
import random
import numpy as np
from peft import PeftModel

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)


BASE = "ku-nlp/deberta-v3-base-japanese"
ADAPTER_REPO = "kkatodus/text_labeller_parliament-deberta-v3-base-japanese"

# Load config from the adapter repo (so num_labels/id2label/label2id come from what you saved)
cfg = AutoConfig.from_pretrained(ADAPTER_REPO)

print(cfg.id2label)
tokenizer = AutoTokenizer.from_pretrained(BASE, use_fast=False, trust_remote_code=False)

base = AutoModelForSequenceClassification.from_pretrained(
    BASE,
    num_labels=cfg.num_labels,          # important
    device_map="auto",
)
base.config.problem_type = "multi_label_classification"
base.config.id2label = cfg.id2label
base.config.label2id = cfg.label2id

model = PeftModel.from_pretrained(base, ADAPTER_REPO)
model.eval()

@torch.no_grad()
def predict(text: str, thr: float = 0.55):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    logits = model(**inputs).logits[0]
    probs = torch.sigmoid(logits).float().cpu().numpy()

    labels = [base.config.id2label[i] for i in range(cfg.num_labels)]
    picked = [labels[i] for i, p in enumerate(probs) if p >= thr]
    if not picked:
        picked = labels[probs.argmax()]
    return probs, picked

probs, picked = predict("私は、お金を使うのもいいけれども、お金を使わないで金融機関が御自分の力で自己資本を充実される仕組みが必要ではないのかなということを常々感じておりました者の一人でございます")
print(picked)

{0: '質問文', 1: '追及・確認文', 2: '答弁文', 3: '反論・再反論文', 4: '意見文', 5: '要望・提案文', 6: '批判文', 7: '評価文', 8: '説明文', 9: '事実文', 10: '謝罪・釈明文', 11: '手続・運営文', 12: 'その他'}


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at ku-nlp/deberta-v3-base-japanese and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


説明文
